### Loading fastANI results data.

In [ ]:
from pathlib import Path
import os
import pandas as pd


intermediates_path = Path("intermediates")
output_path = Path("output")

figures_path = output_path / "figures"

fastani_cwd = intermediates_path / "cwd" / "fastANI"
fastani_results = intermediates_path / "cwd" / "fastANI" / "results" / "fastani_results"

mags_to_cluster_path = intermediates_path / "mags_to_cluster.csv"

per_species_matrices = {
    f.replace("_", " "): pd.read_csv(fastani_results / f, delimiter="\t", names=["query", "reference", "ani", "ortho_found", "ortho_total"]) \
        for f in os.listdir(fastani_results) if (fastani_results / f).is_file()
}
per_species_matrices.keys()

In [ ]:
# MAGs metadata.
mags_to_cluster = pd.read_csv(mags_to_cluster_path, low_memory=False)
mags_to_cluster

### Clustermaps generation.

In [ ]:
from src.figures import create_clustermap_histo_panel, create_four_clustermaps_panel
import numpy as np

# This function excludes comparisons of MAGs from countries other than specified.
def get_geo_filtered_matrix(matrix_df: pd.DataFrame, countries_series: pd.Series, target_geo: str):
    join_st1 = matrix_df.copy().set_index("query").join(countries_series, how="inner", validate="m:1", rsuffix="_query")
    join_st1 = join_st1.rename(columns={"geographic_location": "geographic_location_query"})

    join_st2 = join_st1.reset_index().set_index("reference").join(countries_series, how="inner", validate="m:1", rsuffix="_reference")
    join_st2 = join_st2.rename(columns={"geographic_location": "geographic_location_reference"}).reset_index()

    filtered_matrix_df = join_st2[
        (join_st2["geographic_location_query"] == target_geo) & 
        (join_st2["geographic_location_reference"] == target_geo)]
    filtered_matrix_df = filtered_matrix_df[["query", "reference", "ani"]]
    filtered_matrix_np = filtered_matrix_df.set_index(["query", "reference"])["ani"].unstack().to_numpy()

    return filtered_matrix_np


geo_df = mags_to_cluster.set_index("spire_id")["geographic_location"]


# Generating clustermaps per species.
for species in per_species_matrices.keys():
    species_out_path = figures_path / species
    os.makedirs(species_out_path, exist_ok=True)

    matrix_df = per_species_matrices[species]
    matrix_df = matrix_df[["query", "reference", "ani"]]
    matrix_df["query"] = matrix_df["query"].apply(lambda x: Path(x).stem)
    matrix_df["reference"] = matrix_df["reference"].apply(lambda x: Path(x).stem)

    geo_matrices = [(a, get_geo_filtered_matrix(matrix_df, geo_df, a)) for a in geo_df.sort_values().unique()]

    matrix_np = matrix_df.copy().set_index(["query", "reference"])["ani"].unstack().to_numpy()

    vmin = np.min(matrix_np[np.triu_indices_from(matrix_np)])

    for geoloc, matrix in geo_matrices:
        geo_path = species_out_path / f"{geoloc}.svg"

        create_clustermap_histo_panel(str(geo_path), matrix, vmin)

    create_clustermap_histo_panel(str(species_out_path / "Pooled.svg"), matrix_np, vmin)

    create_four_clustermaps_panel(
        species_out_path / "Multi.svg",
        [a[1] for a in geo_matrices] + [matrix_np],
        [a[0] for a in geo_matrices] + ["Pooled"],
        vmin
    )

    print(f"Processed {species}")

print("Finished.")